In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS hedis.silver;

In [0]:
from pyspark.sql.functions import col, when

patients = spark.table("hedis.bronze.patients")

dim_patient = patients.select(
    col("Id").alias("patient_id"),
    col("BIRTHDATE").alias("birthdate"),
    col("DEATHDATE").alias("deathdate"),
    col("GENDER").alias("gender"),
    col("RACE").alias("race"),
    col("ETHNICITY").alias("ethnicity"),
    col("CITY").alias("city"),
    col("STATE").alias("state"),
    col("ZIP").alias("zip")
).withColumn(
    "region",
    when(col("state") == "Massachusetts", "Northeast")
    .when(col("state") == "California", "West")
    .when(col("state").isin("Texas", "Florida"), "South")
    .otherwise("Unknown")
)

dim_patient.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("hedis.silver.dim_patient")

print(dim_patient.count())

22887


In [0]:
# dim_payer — resolves payer UUIDs to names, flags the uninsured payer for exclusion:

payers = spark.table("hedis.bronze.payers")

dim_payer = payers.select(
    col("Id").alias("payer_id"),
    col("NAME").alias("payer_name")
).withColumn("is_insured", col("payer_name") != "NO_INSURANCE")

dim_payer.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("hedis.silver.dim_payer")

In [0]:
from pyspark.sql.functions import year, to_timestamp

payer_transitions = spark.table("hedis.bronze.payer_transitions")

fact_enrollment = payer_transitions.select(
    col("PATIENT").alias("patient_id"),
    to_timestamp(col("START_DATE"), "yyyy-MM-dd'T'HH:mm:ss'Z'").alias("start_date"),
    to_timestamp(col("END_DATE"), "yyyy-MM-dd'T'HH:mm:ss'Z'").alias("end_date"),
    col("PAYER").alias("payer_id")
).withColumn("start_year", year(col("start_date"))) \
 .withColumn("end_year", year(col("end_date")))

fact_enrollment.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("hedis.silver.fact_enrollment")

In [0]:
conditions = spark.table("hedis.bronze.conditions")

fact_condition = conditions.select(
    col("PATIENT").alias("patient_id"),
    col("ENCOUNTER").alias("encounter_id"),
    col("CODE").alias("condition_code"),
    col("DESCRIPTION").alias("description"),
    col("START").alias("start_date"),
    col("STOP").alias("stop_date")
)

fact_condition.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("hedis.silver.fact_condition")

In [0]:
from pyspark.sql.functions import to_timestamp

observations = spark.table("hedis.bronze.observations")

fact_observation = observations.select(
    col("PATIENT").alias("patient_id"),
    col("ENCOUNTER").alias("encounter_id"),
    to_timestamp(col("DATE"), "yyyy-MM-dd'T'HH:mm:ss'Z'").alias("obs_date"),
    col("CODE").alias("obs_code"),
    col("DESCRIPTION").alias("description"),
    when(col("TYPE") == "numeric", col("VALUE").cast("double")).alias("value_numeric"),
    when(col("TYPE") != "numeric", col("VALUE")).alias("value_text"),
    col("UNITS").alias("units"),
    col("TYPE").alias("value_type")
)

fact_observation.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("hedis.silver.fact_observation")

In [0]:
procedures = spark.table("hedis.bronze.procedures")

fact_procedure = procedures.select(
    col("PATIENT").alias("patient_id"),
    col("ENCOUNTER").alias("encounter_id"),
    col("CODE").alias("procedure_code"),
    col("DESCRIPTION").alias("description"),
    to_timestamp(col("START"), "yyyy-MM-dd'T'HH:mm:ss'Z'").alias("procedure_date"),
    col("REASONCODE").alias("reason_code")
)

fact_procedure.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("hedis.silver.fact_procedure")

In [0]:
systolic = fact_observation.filter(col("obs_code") == "8480-6") \
    .select(col("patient_id"), col("encounter_id"), col("obs_date"), col("value_numeric").alias("systolic"))

diastolic = fact_observation.filter(col("obs_code") == "8462-4") \
    .select(col("encounter_id"), col("value_numeric").alias("diastolic"))

fact_vitals_bp = systolic.join(diastolic, on="encounter_id", how="inner")

fact_vitals_bp.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("hedis.silver.fact_vitals_bp")

print(fact_vitals_bp.count())  # should land close to 12,963 x 4 states if BP volume held proportionally

373233


In [0]:
spark.sql("SHOW TABLES IN hedis.silver").show(truncate=False)
for t in ["dim_patient","dim_payer","fact_enrollment","fact_condition","fact_observation","fact_procedure","fact_vitals_bp"]:
    print(t, spark.table(f"hedis.silver.{t}").count())

+--------+----------------+-----------+
|database|tableName       |isTemporary|
+--------+----------------+-----------+
|silver  |dim_patient     |false      |
|silver  |dim_payer       |false      |
|silver  |fact_condition  |false      |
|silver  |fact_enrollment |false      |
|silver  |fact_observation|false      |
|silver  |fact_procedure  |false      |
|silver  |fact_vitals_bp  |false      |
+--------+----------------+-----------+

dim_patient 22887
dim_payer 40
fact_enrollment 840179
fact_condition 819638
fact_observation 17137977
fact_procedure 3667575
fact_vitals_bp 373233


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

w = Window.partitionBy("encounter_id").orderBy(col("obs_date"))

systolic = fact_observation.filter(col("obs_code") == "8480-6") \
    .withColumn("rn", row_number().over(w)).filter(col("rn") == 1) \
    .select(col("patient_id"), col("encounter_id"), col("obs_date"), col("value_numeric").alias("systolic"))

diastolic = fact_observation.filter(col("obs_code") == "8462-4") \
    .withColumn("rn", row_number().over(w)).filter(col("rn") == 1) \
    .select(col("encounter_id"), col("value_numeric").alias("diastolic"))

fact_vitals_bp = systolic.join(diastolic, on="encounter_id", how="inner")

fact_vitals_bp.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("hedis.silver.fact_vitals_bp")

print(fact_vitals_bp.count())

324390
